# Demonstration: Common Testing Strategies

In this demonstration, you will investigate common pipeline testing strategies for data science and machine learning. In addition, you will be given an introduction to `unittest`, a common testing framework. You will gain a new perspective of MLflow as an integral tool for proper model testing and important to modern MLOps pipelines.

---

### Learning Objectives

By the end of this demonstration, you will be able to do the following:

* **Understand the differences between different pipeline tests** and how to build helper functions to validate the following types of tests:
  * **Data Validation**
  * **Data Transformation**
  * **Model Integration**
  * **Modeling Functions**
    * *Testing model functions will utilize MLflow and Unity Catalog for comprehensive versioning and lineage.*
* **Understand `unittest`** as a comprehensive testing framework.

> ⚠️ **Warning:** Some of the cells are meant to fail for demonstration purposes.

---

### Requirements

Please review the following requirements before starting the lesson:

* **To run this notebook, you need to use one of the following Databricks runtime(s):** `15.4.x-cpu-ml-scala2.12`

## Requirements

Please review the following requirements before starting the lesson:

* **To run this notebook, you need to use one of the following Databricks runtime(s):** `15.4.x-cpu-ml-scala2.12`

---

## REQUIRED - SELECT CLASSIC COMPUTE

Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default. Follow these steps to select the classic compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.
2. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:
   * In the drop-down, select **More**.
   * In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.
2. Find the triangle icon to the right of your compute cluster name and click it.
3. Wait a few minutes for the cluster to start.
4. Once the cluster is running, complete the steps above to select your cluster.

---

## Classroom Setup

To get into the lesson, we first need to build some data assets and define some configuration variables required for this demonstration. When running the following cell, the output is hidden so our space isn't cluttered. To view the details of the output, you can hover over the next cell and click the eye icon.

The cell after the setup, titled `View Setup Variables`, displays the various variables that were created. You can click the Catalog icon in the notebook space to the right to see that your catalog was created with no data.

In [0]:
#%run ../Includes/Classroom-Setup-Demo-M02

## Further Preparation - Train, Register, and Serve ML model

For testing that our model is behaving as intended, we will train and package our model with MLflow, register it to Unity Catalog, serve the model using Mosaic AI Model Serving.

**Warning:** It will take a few minutes to setup the Model Serving Endpoint.

In [0]:
#print(f"Username:          {DA.username}")
#print(f"Catalog Name:      {DA.catalog_name}")
#print(f"Schema Name:       {DA.schema_name}")
#print(f"Working Directory: {DA.paths.working_dir}")
#print(f"Dataset Location:  {DA.paths.datasets}")

username = 'workspace'
catalog_name = 'default'
#schema_name =''

print(f"Username:          {username}")
print(f"Catalog Name:      {catalog_name}")
#print(f"Schema Name:       {schema_name}")
#print(f"Working Directory: {paths.working_dir}")
#print(f"Dataset Location:  {paths.datasets}")

Username:          workspace
Catalog Name:      default


In [0]:
# Modify the registry uri to point to Unity Catalog
import mlflow
mlflow.set_registry_uri("databricks-uc")

In [0]:
#arquivo rich_pedboard
schema_name ='rich_pedboard'
# Define the model name
model_name = f"{catalog_name}.{schema_name}.{username}"
print(model_name)

default.rich_pedboard.workspace


In [0]:
import mlflow
from mlflow.models.signature import infer_signature

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

from pyspark.sql import SparkSession

# Start a Spark session
spark = SparkSession.builder.getOrCreate()


In [0]:
import pandas as pd
# 1. URL pública válida contendo o mesmo arquivo 'diabetes_prediction_dataset.csv'
csv_url = "https://raw.githubusercontent.com/NANITH777/Diabetes-Prediction-ID3_Alg-ML-Models/main/diabetes_prediction_dataset.csv"

# 2. Carrega o CSV via Pandas
df_raw = pd.read_csv(csv_url)

In [0]:
if "id" not in df_raw.columns:
    df_raw["id"] = df_raw.index

In [0]:
# 3. Converte para Spark DataFrame e salva a tabela Delta
df_spark = spark.createDataFrame(df_raw)
df_spark.write.format("delta").mode("overwrite").saveAsTable("diabetes_new")

{"ts": "2026-08-17 00:23:43.565", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzQyOTI0NDI5Mzg0MjgwNRABIAEyJDAxYTAwZDE5LWQ1YjUtN2FlNS1iODJhLTk0ZjRkOWMyMjAzYTokYjRkNGM2MjYtNmUxMi0zYzI2LTg1ZTMtODVhMDM3NmE2OTkxSgsI8KiJ1AYQwO6iGVABWAFgAWj6iMC6lsWjDQ==.", "context": {}}
{"ts": "2026-08-17 00:23:43.565", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzQyOTI0NDI5Mzg0MjgwNRABIAEyJDAxYTAwZDE5LWQ1YjUtN2FlNS1iODJhLTk0ZjRkOWMyMjAzYTokYjRkNGM2MjYtNmUxMi0zYzI2LTg1ZTMtODVhMDM3NmE2OTkxSgsI8KiJ1AYQwO6iGVABWAFgAWj6iMC6lsWjDQ==.", "context": {}}
{"ts": "2026-08-17 00:23:43.565", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzQyOTI0NDI5Mzg0MjgwNRABIAEyJDAxYTAwZDE5LWQ1YjUtN2FlNS1iODJhLTk0ZjRkOWMyMjAzYTokYjRkNGM2MjYtNmUxMi0zYzI2LTg1ZTMtODVhMDM3NmE2OTkx

In [0]:


# Load dataset
df = spark.read.format('delta').table('diabetes_new')
training_df = df.toPandas()


In [0]:
training_df.tail()

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes,id
99995,Male,19.0,0,0,No Info,27.32,6.5,140,0,12495
99996,Female,80.0,1,0,never,25.68,5.0,85,0,12496
99997,Female,31.0,0,0,No Info,30.27,6.0,160,0,12497
99998,Female,80.0,0,0,No Info,28.20,8.8,159,1,12498
99999,Male,37.0,0,0,never,27.90,6.2,85,0,12499


In [0]:
df = df.withColumnRenamed("diabetes", "Diabetes_binary")

In [0]:
training_df = df.toPandas()

In [0]:
training_df.tail()

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,Diabetes_binary,id
99995,Male,19.0,0,0,No Info,27.32,6.5,140,0,12495
99996,Female,80.0,1,0,never,25.68,5.0,85,0,12496
99997,Female,31.0,0,0,No Info,30.27,6.0,160,0,12497
99998,Female,80.0,0,0,No Info,28.20,8.8,159,1,12498
99999,Male,37.0,0,0,never,27.90,6.2,85,0,12499


In [0]:

X = training_df.drop(["id", "Diabetes_binary"], axis=1)
y = training_df["Diabetes_binary"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [0]:
username

'workspace'

In [0]:
# set the path for mlflow experiment
# Use the actual user directory path
actual_user = "fabieneaulas@gmail.com"
mlflow.set_experiment(f"/Users/{actual_user}/{username}_model")

with mlflow.start_run(run_name = 'mlflow-run') as run:
    # Initialize the Random Forest classifier
    rf_classifier = RandomForestClassifier(random_state=42)
    
    # Fit the model on the training data
    X_train_encoded = pd.get_dummies(X_train) # adaptado
    rf_classifier.fit(X_train_encoded, y_train)

    # Encode test data with the same columns as training data
    X_test_encoded = pd.get_dummies(X_test)
    # Align test set columns with training set columns
    X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)
    
    # Make predictions on the test data
    y_pred = rf_classifier.predict(X_test_encoded)
    
    # Enable automatic logging of input samples, metrics, parameters, and models
    mlflow.sklearn.autolog(
        log_input_examples = True,
        silent = True
    )


    mlflow.sklearn.log_model(
        rf_classifier,
        artifact_path = "model-artifacts",
        input_example = X_train_encoded[:3],
        signature = infer_signature(X_train_encoded, y_train)
    )

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/08/17 00:23:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-0b69ef93-5247.cloud.databricks.com/ml/experiments/3429244293842807/models/m-298fee25b514494f8cd7b

In [0]:


model_uri = f"runs:/{run.info.run_id}/model-artifacts"



In [0]:
model_uri

'runs:/59317ebe56b645848389f38fe098e719/model-artifacts'

In [0]:
%sql
SHOW CATALOGS


catalog
samples
system
workspace


In [0]:
schema_name

'rich_pedboard'

In [0]:
schema_name = 'default'

## Register the Model with UC


In [0]:
# Register the model in the model registry
# Use 'workspace' catalog instead of 'default' which doesn't exist
registered_model = mlflow.register_model(model_uri=model_uri, name=f"workspace.{schema_name}.testing_strats_model")

Registered model 'workspace.default.testing_strats_model' already exists. Creating a new version of this model...
2026/08/17 00:24:15 WARNING mlflow.tracking._model_registry.fluent: Run with id 59317ebe56b645848389f38fe098e719 has no artifacts at artifact path 'model-artifacts', registering model based on models:/m-298fee25b514494f8cd7b1315b919f2b instead


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '5' of model 'workspace.default.testing_strats_model': https://dbc-0b69ef93-5247.cloud.databricks.com/explore/data/models/workspace/default/testing_strats_model/version/5?o=7474657872577658


# Delete old endpoint

In [0]:
from databricks.sdk import WorkspaceClient

try:
    # Initialize the workspace client
    workspace = WorkspaceClient()

    # Delete the serving endpoint
    workspace.serving_endpoints.delete(name=f"M02-endpoint_{schema_name}")
    print('Deleted Endpoint M02-endpoint')
except:
    print(f"Endpoint M02-endpoint_{schema_name} does not exist.")

Endpoint M02-endpoint_default does not exist.


In [0]:
from databricks.sdk import WorkspaceClient
schema_name_2= 'rich_pedboard'
try:
    # Initialize the workspace client
    workspace = WorkspaceClient()

    # Delete the serving endpoint
    workspace.serving_endpoints.delete(name=f"M02-endpoint_{schema_name}")
    print('Deleted Endpoint M02-endpoint')
except:
    print(f"Endpoint M02-endpoint_{schema_name} does not exist.")


Endpoint M02-endpoint_default does not exist.


### Serve the model (Início)

In [0]:
schema_name

'default'

In [0]:
catalog_name ='workspace'

In [0]:
from mlflow.deployments import get_deploy_client

client = get_deploy_client("databricks")
endpoint_name = f"M02-endpoint_{schema_name_2}"
endpoint_name = endpoint_name.replace("@databricks.com", "").replace('.', '-')
spark.sql(f'use catalog {catalog_name}')
spark.sql(f'use schema {schema_name}')
# Check if the endpoint already exists

DataFrame[]

In [0]:

try:
    # Attempt to get the endpoint
    existing_endpoint = client.get_endpoint(endpoint_name)
    print(f"Endpoint '{endpoint_name}' already exists.")
except Exception as e:
    # If not found, create the endpoint
    if "RESOURCE_DOES_NOT_EXIST" in str(e):
        print(f"Creating a new endpoint: {endpoint_name}")
        endpoint = client.create_endpoint(
            name=endpoint_name,
            config={
                "served_entities": [
                    {
                        "name": "strats-model",
                        "entity_name": f"{catalog_name}.{schema_name}.testing_strats_model",
                        "entity_version": 1,
                        "workload_size": "Small",
                        "scale_to_zero_enabled": True
                    }
                ],
                "traffic_config": {
                    "routes": [
                        {
                            "served_model_name": "strats-model",
                            "traffic_percentage": 100
                        }
                    ]
                }
            }
        )
        
    else:
        print(f"An error occurred: {e}")

Endpoint 'M02-endpoint_rich_pedboard' already exists.


## Common ML Pipeline Testing

Here we will go over common ML pipeline testing paradigms that can be used to ensure robust and reliable ML models by testing various aspects of the ML pipeline. We will focus on the following:

1. **Data Validation** - data quality checks, expected pattern checks, and custom business logic are enforced.
2. **Data Transformations** - apply data transformations like normalization and encoding for feature engineering tasks.
3. **Modeling functions** - unit and integration tests are used to test individual component interactions as well as overall end-to-end testing.
4. **Model Integration** - CI/CD integration along with MLOps workflows to enable long-term efficiency of ML systems

When exploring each of these components of testing, we will provide helper functions. It should be noted that this is not a comprehensive dive into each of these topics.

---

### Data Validation

Data Validation for ML Pipeline Testing involves ensuring that the input, intermediate, and output data in a machine learning pipeline meet expected quality, structure, and distribution standards. It is a critical step to verify that the data being processed in an ML pipeline aligns with the assumptions made during model development, ensuring the reliability and correctness of the pipeline. Here we will verify our dataframe schema, that no values missing, and that non-negative values do not exist.

---



## Define Helper Functions

These functions provide validation checks for (PySpark) DataFrames to ensure data quality and consistency.

* `validate_schema` ensures the schema matches an expected definition.
* `validate_no_missing_values` checks for and reports any missing (null) values in the DataFrame.
* `validate_binary_column` confirms that a specified column contains only binary values (0 or 1).
* `validate_double_column` verifies that a specific column's data type is double.
* `validate_non_negative_values` ensures that all columns in the DataFrame contain only non-negative values.

Overall, these function help to enforce data integrity for downstream machine learning or analytical tasks.

In [0]:
df.schema

StructType([StructField('gender', StringType(), True), StructField('age', DoubleType(), True), StructField('hypertension', LongType(), True), StructField('heart_disease', LongType(), True), StructField('smoking_history', StringType(), True), StructField('bmi', DoubleType(), True), StructField('HbA1c_level', DoubleType(), True), StructField('blood_glucose_level', LongType(), True), StructField('Diabetes_binary', LongType(), True), StructField('id', LongType(), True)])

In [0]:
df.printSchema()

root
 |-- gender: string (nullable = true)
 |-- age: double (nullable = true)
 |-- hypertension: long (nullable = true)
 |-- heart_disease: long (nullable = true)
 |-- smoking_history: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- HbA1c_level: double (nullable = true)
 |-- blood_glucose_level: long (nullable = true)
 |-- Diabetes_binary: long (nullable = true)
 |-- id: long (nullable = true)



In [0]:
def validate_schema(df, expected_schema):
    """
    Validates whether the schema of the given DataFrame matches the expected schema.

    Parameters:
    - df: The PySpark DataFrame to check.
    - expected_schema: The expected schema (StructType).

    Uses:
    - AssertionError if the schema does not match.
    """

    actual_schema = df.schema
    assert actual_schema == expected_schema, (
        f"Schema validation failed.\n"
        f"Expected schema: {expected_schema}\n"
        f"Actual schema: {actual_schema}"
    )
    print("Schema validation passed!")


def validate_no_missing_values(df):
    """
    Validates that the given PySpark DataFrame contains no missing (null) values.

    Parameters:
    - df: The PySpark DataFrame to check.

    Uses:
    - AssertionError if missing values are found.
    """
    from pyspark.sql.functions import col, sum

    missing_values = df.agg(*[
        sum(col(c).isNull().cast("int")).alias(c) for c in df.columns
    ]).collect()[0].asDict()

    missing_columns = {col: missing_values[col] for col in df.columns if missing_values[col] > 0}

    assert not missing_columns, (
        f"Missing values found in the following columns: {missing_columns}"
    )
    print("No missing values detected!")



def validate_binary_column(df, column_name):
    """
    Validates that the specified column in the DataFrame contains only binary values (0 or 1).

    Parameters:
    - df: The PySpark DataFrame to check.
    - column_name: The name of the column to validate.

    Uses:
    - AssertionError if the column contains non-binary values.
    """
    # Find distinct values in the column
    distinct_values = df.select(column_name).distinct().rdd.flatMap(lambda x: x).collect()

    # Check if all distinct values are in the set {0, 1}
    is_binary = set(distinct_values).issubset({0, 1})

    assert is_binary, (
        f"Column '{column_name}' contains non-binary values: {set(distinct_values)}"
    )
    print(f"Column '{column_name}' is binary (0 or 1).")


def validate_double_column(df, column_name):
    """
    Validates that the specified column in the DataFrame is of type double.

    Parameters:
    - df: The PySpark DataFrame to check.
    - column_name: The name of the column to validate.

    Uses:
    - AssertionError if the column is not of type double.
    """
    # Get the data type of the column
    column_data_type = df.schema[column_name].dataType

    # Check if the data type is DoubleType
    is_double = isinstance(column_data_type, DoubleType)

    assert is_double, (
        f"Column '{column_name}' is not of type double. Found type: {column_data_type}"
    )
    print(f"Column '{column_name}' is of type double.")

    
def validate_non_negative_values(df):
    """
    Validates that all columns in the given PySpark DataFrame contain non-negative values (>= 0).

    Parameters:
    - df: The PySpark DataFrame to check.

    Uses:
    - AssertionError if any column contains negative values.
    """
    from pyspark.sql.functions import col, min

    negative_values = df.agg(*[
        min(col(c)).alias(c) for c in df.columns
    ]).collect()[0].asDict()

    negative_columns = {col: negative_values[col] for col in df.columns if negative_values[col] < 0}

    assert not negative_columns, (
        f"Negative values found in the following columns: {negative_columns}"
    )
    print("All columns contain non-negative values!")

# Define expected schema for our DataFrame

In [0]:
from pyspark.sql.types import StructType, StructField, DoubleType, LongType, IntegerType

# Define the expected schema
expected_schema = StructType([
    StructField("id", LongType(), True),
    StructField("Diabetes_binary", IntegerType(), True),
    StructField("HighBP", IntegerType(), True),
    StructField("BMI", IntegerType(), True),
    StructField("Smoker", IntegerType(), True),
    StructField("Stroke", IntegerType(), True),
    StructField("HeartDiseaseorAttack", IntegerType(), True),
    StructField("Age", DoubleType(), True),
])

In [0]:
# Run validation tests
#validate_schema(df, expected_schema)
validate_no_missing_values(df)
#validate_binary_column(df, "HeartDiseaseorAttack")
#validate_double_column(df, "Age")
#validate_non_negative_values(df)

No missing values detected!


Oh no! We seem to have an AssertionError indicating there's been some faulty entry. It looks like there is a record within the HeartDiseaseorAttack feature that has a value of -1. After talking to the data team, it turns out there was some faulty logic (a typo if you can believe it) that converted this to -1 from 1. Let's clean it this up and continue our validations.

In [0]:
df.show(5)

+------+----+------------+-------------+---------------+-----+-----------+-------------------+---------------+---+
|gender| age|hypertension|heart_disease|smoking_history|  bmi|HbA1c_level|blood_glucose_level|Diabetes_binary| id|
+------+----+------------+-------------+---------------+-----+-----------+-------------------+---------------+---+
|Female|80.0|           0|            1|          never|25.19|        6.6|                140|              0|  0|
|Female|54.0|           0|            0|        No Info|27.32|        6.6|                 80|              0|  1|
|  Male|28.0|           0|            0|          never|27.32|        5.7|                158|              0|  2|
|Female|36.0|           0|            0|        current|23.45|        5.0|                155|              0|  3|
|  Male|76.0|           1|            1|        current|20.14|        4.8|                155|              0|  4|
+------+----+------------+-------------+---------------+-----+-----------+------

In [0]:
from pyspark.sql.functions import when
# Convert values in HeartDiseaseorAttack that are -1 to 1
#df = df.withColumn("HeartDiseaseorAttack", when(df.HeartDiseaseorAttack == -1, 1).otherwise(df.HeartDiseaseorAttack))

In [0]:
#validate_binary_column(df, "HeartDiseaseorAttack")
validate_double_column(df, "age")


Column 'age' is of type double.


In [0]:
#df_encoded = pd.get_dummies(df) # adaptado

In [0]:
import gc

# 1. Remova as variáveis de modelos que você não precisa mais
#del pipeline, df_encoded  # substitua pelos nomes das suas variáveis de modelo/pipeline

# 2. Force o Garbage Collector do Python a limpar a memória
gc.collect()

116733

In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline

# Definindo explicitamente as colunas categóricas
categorical_cols = ["gender", "smoking_history"]

# Criando os estágios para indexação e codificação
indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_indexed", handleInvalid="keep")
    for col in categorical_cols
]
encoders = [
    OneHotEncoder(inputCol=f"{col}_indexed", outputCol=f"{col}_encoded")
    for col in categorical_cols
]

# Ajustando e transformando o DataFrame
pipeline = Pipeline(stages=indexers + encoders)
df_encoded = pipeline.fit(df).transform(df)

# Colunas intermediárias e originais a serem removidas
cols_to_drop = categorical_cols + [f"{col}_indexed" for col in categorical_cols]

# Remove as colunas antigas e mantém apenas as versões _encoded
df_final = df_encoded.drop(*cols_to_drop)

display(df_final.show(5))

+----+------------+-------------+-----+-----------+-------------------+---------------+-----+--------------+-----------------------+
| age|hypertension|heart_disease|  bmi|HbA1c_level|blood_glucose_level|Diabetes_binary|   id|gender_encoded|smoking_history_encoded|
+----+------------+-------------+-----+-----------+-------------------+---------------+-----+--------------+-----------------------+
|44.0|           0|            0|42.25|        6.2|                155|              0|25000| (3,[0],[1.0])|          (6,[3],[1.0])|
|49.0|           0|            0| 27.6|        6.0|                126|              0|25001| (3,[0],[1.0])|          (6,[2],[1.0])|
|58.0|           0|            0|38.24|        5.8|                 85|              0|25002| (3,[0],[1.0])|          (6,[1],[1.0])|
|57.0|           1|            0|22.95|        6.0|                 90|              0|25003| (3,[0],[1.0])|          (6,[1],[1.0])|
|66.0|           0|            0|33.76|        5.7|                10

In [0]:
#from pyspark.ml.feature import StringIndexer, OneHotEncoder
#from pyspark.ml import Pipeline

# Identify categorical columns (excluding 'id')
#categorical_cols = [c for c, t in df.dtypes if t == 'string' and c != 'id']

# StringIndexer and OneHotEncoder for each categorical column
#indexers = [StringIndexer(inputCol=col, outputCol=f"{col}_indexed") for col in categorical_cols]
#encoders = [OneHotEncoder(inputCol=f"{col}_indexed", outputCol=f"{col}_encoded") for col in categorical_cols]

#pipeline = Pipeline(stages=indexers + encoders)
#df_encoded = pipeline.fit(df).transform(df)

#display(df_encoded)

In [0]:
df_encoded.printSchema()


root
 |-- gender: string (nullable = true)
 |-- age: double (nullable = true)
 |-- hypertension: long (nullable = true)
 |-- heart_disease: long (nullable = true)
 |-- smoking_history: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- HbA1c_level: double (nullable = true)
 |-- blood_glucose_level: long (nullable = true)
 |-- Diabetes_binary: long (nullable = true)
 |-- id: long (nullable = true)
 |-- gender_indexed: double (nullable = false)
 |-- smoking_history_indexed: double (nullable = false)
 |-- gender_encoded: vectorudt (nullable = true)
 |-- smoking_history_encoded: vectorudt (nullable = true)



In [0]:
# Filter to numeric columns only for validation
from pyspark.sql.types import NumericType
numeric_cols = [field.name for field in df_encoded.schema.fields if isinstance(field.dataType, NumericType)]
df_numeric = df_encoded.select(*numeric_cols)
validate_non_negative_values(df_numeric)

All columns contain non-negative values!


In [0]:
df_numeric.show(5)

+----+------------+-------------+-----+-----------+-------------------+---------------+-----+--------------+-----------------------+
| age|hypertension|heart_disease|  bmi|HbA1c_level|blood_glucose_level|Diabetes_binary|   id|gender_indexed|smoking_history_indexed|
+----+------------+-------------+-----+-----------+-------------------+---------------+-----+--------------+-----------------------+
|44.0|           0|            0|42.25|        6.2|                155|              0|25000|           0.0|                    3.0|
|49.0|           0|            0| 27.6|        6.0|                126|              0|25001|           0.0|                    2.0|
|58.0|           0|            0|38.24|        5.8|                 85|              0|25002|           0.0|                    1.0|
|57.0|           1|            0|22.95|        6.0|                 90|              0|25003|           0.0|                    1.0|
|66.0|           0|            0|33.76|        5.7|                10

In [0]:
df.show(5)

+------+----+------------+-------------+---------------+-----+-----------+-------------------+---------------+---+
|gender| age|hypertension|heart_disease|smoking_history|  bmi|HbA1c_level|blood_glucose_level|Diabetes_binary| id|
+------+----+------------+-------------+---------------+-----+-----------+-------------------+---------------+---+
|Female|80.0|           0|            1|          never|25.19|        6.6|                140|              0|  0|
|Female|54.0|           0|            0|        No Info|27.32|        6.6|                 80|              0|  1|
|  Male|28.0|           0|            0|          never|27.32|        5.7|                158|              0|  2|
|Female|36.0|           0|            0|        current|23.45|        5.0|                155|              0|  3|
|  Male|76.0|           1|            1|        current|20.14|        4.8|                155|              0|  4|
+------+----+------------+-------------+---------------+-----+-----------+------

In [0]:
df_encoded.schema.fields

[StructField('gender', StringType(), True),
 StructField('age', DoubleType(), True),
 StructField('hypertension', LongType(), True),
 StructField('heart_disease', LongType(), True),
 StructField('smoking_history', StringType(), True),
 StructField('bmi', DoubleType(), True),
 StructField('HbA1c_level', DoubleType(), True),
 StructField('blood_glucose_level', LongType(), True),
 StructField('Diabetes_binary', LongType(), True),
 StructField('id', LongType(), True),
 StructField('gender_indexed', DoubleType(), False),
 StructField('smoking_history_indexed', DoubleType(), False),
 StructField('gender_encoded', VectorUDT(), True),
 StructField('smoking_history_encoded', VectorUDT(), True)]

In [0]:
# Filter to numeric columns only for validation
from pyspark.sql.types import NumericType
numeric_cols = [field.name for field in df.schema.fields if isinstance(field.dataType, NumericType)]
numeric_cols

['age',
 'hypertension',
 'heart_disease',
 'bmi',
 'HbA1c_level',
 'blood_glucose_level',
 'Diabetes_binary',
 'id']

In [0]:
df.schema.fields

[StructField('gender', StringType(), True),
 StructField('age', DoubleType(), True),
 StructField('hypertension', LongType(), True),
 StructField('heart_disease', LongType(), True),
 StructField('smoking_history', StringType(), True),
 StructField('bmi', DoubleType(), True),
 StructField('HbA1c_level', DoubleType(), True),
 StructField('blood_glucose_level', LongType(), True),
 StructField('Diabetes_binary', LongType(), True),
 StructField('id', LongType(), True)]

In [0]:

df_numeric_2 = df.select(*numeric_cols)
validate_non_negative_values(df_numeric_2)

All columns contain non-negative values!


In [0]:
df_numeric_2.show(7)

+----+------------+-------------+-----+-----------+-------------------+---------------+---+
| age|hypertension|heart_disease|  bmi|HbA1c_level|blood_glucose_level|Diabetes_binary| id|
+----+------------+-------------+-----+-----------+-------------------+---------------+---+
|80.0|           0|            1|25.19|        6.6|                140|              0|  0|
|54.0|           0|            0|27.32|        6.6|                 80|              0|  1|
|28.0|           0|            0|27.32|        5.7|                158|              0|  2|
|36.0|           0|            0|23.45|        5.0|                155|              0|  3|
|76.0|           1|            1|20.14|        4.8|                155|              0|  4|
|20.0|           0|            0|27.32|        6.6|                 85|              0|  5|
|44.0|           0|            0|19.31|        6.5|                200|              1|  6|
+----+------------+-------------+-----+-----------+-------------------+---------

In [0]:
# usar o df_numeric

## Data Transformations

Transformation testing involved validating and verifying the operations applied to raw data to prepare it for use in a machine learning pipeline. These transformations include cleaning, scaling, encoding, and feature extraction, which are critical for ensuring that the data fed into the model aligns with the intended assumptions. Here we will consider performing a normalization test on the `Age` feature and validating that the transformation has been applied correctly.

---

### Define Helper Functions

These helper functions ensure proper normalization of a DataFrame column and validate the process.

* `normalize_column` creates a normalized version of a specified column by subtracting its mean and dividing by its standard deviation, appending the result as a new column in the DataFrame.
* `test_column_normalized` verifies the correctness of the normalization by checking if the resulting column's mean is approximately 0 and its standard deviation is approximately 1, within a small tolerance, to confirm accurate scaling for downstream analysis or modeling.

In [0]:
# This function will normalize the dataframe's column
def normalize_column(df, column):
    if column not in df.columns:
        raise AssertionError(f"Column '{column}' does not exist in the DataFrame.")
    
    # Simulate a normalized column for demonstration
    df[f'{column}_normalized'] = (df[column] - df[column].mean()) / df[column].std()
    return df

# Test function to check normalization
def test_column_normalized(df, column):
    if column not in df.columns:
        raise AssertionError(f"Column '{column}' does not exist in the DataFrame.")
    
    mean = np.mean(df[column])
    std = np.std(df[column])
    
    # Allowing a small tolerance for floating-point arithmetic
    tolerance = 1e-4
    assert abs(mean) < tolerance, f"Mean of column '{column}' is not approximately 0. It is {mean}."
    assert abs(std - 1) < tolerance, f"Standard deviation of column '{column}' is not approximately 1. It is {std}."
    print(f"Column '{column}' is properly normalized.")

# Normalization Test

In [0]:
df = spark.read.format('delta').table('diabetes_new').toPandas()
df = normalize_column(df, 'age')

In [0]:
import numpy as np

In [0]:

test_column_normalized(df, 'age_normalized')

Column 'age_normalized' is properly normalized.


## Model Integration

Copy and paste the url for the model serving endpoint. Locate your endpoint under **Serving** and click on it. Then copy the **URL**. Paste it in the following cell.

Clicar em serving e visualiza o notebook

![image_1786923024286.png](./image_1786923024286.png "image_1786923024286.png")

![image_1786923099804.png](./image_1786923099804.png "image_1786923099804.png")

# copiar a url da página 

In [0]:
# 32: Grab the API URL and TOKEN

import os
# Retrieve the API URL and token using dbutils
#https://dbc-0b69ef93-5247.cloud.databricks.com/serving-endpoints/M02-endpoint_rich_pedboard/invocations
API_URL =  'https://dbc-0b69ef93-5247.cloud.databricks.com/serving-endpoints/M02-endpoint_rich_pedboard/invocations'
#API_URL = <copy_and_paste>
TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().getOrElse(None)

## Define Helper Functions

The function, `test_model_endpoint_status`, checks whether a model serving endpoint is operational by sending a test dataset as a request and verifying that it returns a 200 status code. It prepares the input dataset in the expected JSON format, sends it to the endpoint using a POST request with appropriate headers, and evaluates the response. If the status code is 200, it confirms the endpoint is functioning correctly; otherwise, it logs the error or response details. This function ensures the endpoint is ready to handle requests for predictions.

In [0]:
import requests
import json

def test_model_endpoint_status(dataset, url):
    """
    Test if the model serving endpoint returns a 200 status code.

    Args:
        dataset (pd.DataFrame): Input dataset to send to the model endpoint.
        url (string): The URL of the model serving endpoint.

    Returns:
        None. Prints the result of the test.
    """
    try:
        # Make a request to the model endpoint using the score_model function
        headers = {'Authorization': f'Bearer {TOKEN}', 'Content-Type': 'application/json'}

        # Prepare the data in the expected format
        ds_dict = {'dataframe_split': dataset.to_dict(orient='split')}
        data_json = json.dumps(ds_dict, allow_nan=True)

        # Send the request
        response = requests.request(method='POST', headers=headers, url=url, data=data_json)

        # Check the status code
        if response.status_code == 200:
            print("Test passed: Endpoint returned status code 200.")
        else:
            print(f"Test failed: Endpoint returned status code {response.status_code}. Response: {response.text}")
    except Exception as e:
        print(f"Test failed with error: {e}")

In [0]:
# grab the first row of the pandas dataframe X_test
X_test.iloc[[0]]

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level
75721,Male,16.0,0,0,never,26.04,6.0,130


In [0]:
# Update the endpoint to use the new model version
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

workspace = WorkspaceClient()

# Define the endpoint configuration to use model version 3
config = EndpointCoreConfigInput(
    served_entities=[
        ServedEntityInput(
            entity_name="workspace.default.testing_strats_model",
            entity_version="3",
            workload_size="Small",
            scale_to_zero_enabled=True
        )
    ]
)

endpoint_name_update = "M02-endpoint_rich_pedboard"

# Update the endpoint
workspace.serving_endpoints.update_config(
    name=endpoint_name_update,
    served_entities=config.served_entities
)

print(f"Endpoint '{endpoint_name_update}' updated to use model version 3")

Endpoint 'M02-endpoint_rich_pedboard' updated to use model version 3


In [0]:
# Encode test data the same way as training data
# This must match the encoding used when training the model in Cell 20
X_train_encoded = pd.get_dummies(X_train)
X_test_encoded = pd.get_dummies(X_test)
# Align test set columns with training set columns
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

print(f"X_test_encoded shape: {X_test_encoded.shape}")
print(f"Columns: {X_test_encoded.columns.tolist()[:10]}...")  # Show first 10 columns

X_test_encoded shape: (20000, 15)
Columns: ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'gender_Female', 'gender_Male', 'gender_Other', 'smoking_history_No Info']...


Note that the following code might not run quickly since we set up our endpoint to scale to zero. If we run it again we will see much small latency.

In [0]:
# Call the test function in your Databricks notebook
test_model_endpoint_status(X_test_encoded.iloc[[0]], url = API_URL)

Test passed: Endpoint returned status code 200.


Model testing involves training, prediction, and evaluation of ML models as a part of the ML pipeline. Unit tests and integration tests are foundational components and should be considered separately. Here we will consider inference as an example to determine if the F1-score and accuracy are what we expect.

The following function, `test_model_performance`, evaluates a logged MLflow model's performance on a test dataset by calculating its F1-score and accuracy. It loads the model using its URI, generates predictions on the test features, and compares the calculated metrics against specified thresholds. If the F1-score or accuracy falls below the thresholds, the function raises an assertion error; otherwise, it confirms the model meets the performance criteria and prints the metrics. This ensures the model's predictions align with expected performance levels before deployment or further use.

In [0]:
import mlflow.pyfunc
from sklearn.metrics import f1_score, accuracy_score

def test_model_performance(model_uri, X_test, y_test, f1_threshold=0.7, accuracy_threshold=0.8):
    """
    Test function to evaluate a logged MLflow model on a test dataset.

    Args:
        model_uri (str): URI of the logged MLflow model.
        X_test (pd.DataFrame): Test features.
        y_test (pd.Series): True labels for the test set.
        f1_threshold (float): Minimum acceptable F1-score.
        accuracy_threshold (float): Minimum acceptable accuracy score.

    Raises:
        AssertionError: If model performance metrics do not meet expected thresholds.
    """
    # Step 1: Load the model from MLflow
    model = mlflow.pyfunc.load_model(model_uri)

    # Step 2: Make predictions on the test set
    y_pred = model.predict(X_test_encoded) # trocar X_test para X_test_encoded

    # Step 3: Calculate evaluation metrics
    f1 = f1_score(y_test, y_pred) 
    accuracy = accuracy_score(y_test, y_pred)

    # Step 4: Assert performance thresholds
    assert f1 >= f1_threshold, f"F1-score {f1:.4f} is below the threshold of {f1_threshold:.2f}."
    assert accuracy >= accuracy_threshold, f"Accuracy {accuracy:.4f} is below the threshold of {accuracy_threshold:.2f}."

    # Step 5: Print metrics and success message
    print("Model performance passed all thresholds.")
    print(f"F1-score: {f1:.4f}, Accuracy: {accuracy:.4f}")

In [0]:
# Example usage of the test function
try:
    test_model_performance(
        model_uri=model_uri,
        X_test=X_test,
        y_test=y_test,
        f1_threshold=0.7, # Custom threshold for F1-score
        accuracy_threshold=0.7 # Custom threshold for accuracy
    )
except AssertionError as e:
    print(e)

Model performance passed all thresholds.
F1-score: 0.7932, Accuracy: 0.9701


## Testing Frameworks

Unit tests are essential in the ML development process. They verify that individual components of your code. For example, it's common practice to check that functions or methods function as intended by data scientists and machine learning practitioners.

---

### Unittest

unittest involves testing individual components of an ML pipeline to ensure correctness and reliability. Since machine learning pipelines consist of multiple stages—data preprocessing, feature engineering, model training, evaluation, and inference—unit testing helps validate the behavior of each component in isolation. Here we will consider unit tests for schema vliadation and checking missing values.

**Why use unittest?**

Using unittest is better than writing individual functions for testing because it provides a structured, standardized, and scalable framework for managing and executing tests.

---

### Validating Schema and Missing and Non-negative Values

Here we will return back to our first example and use unittest for data validation. Here, we will validate the schema, that no missing values are present, and non-negative values are not present.

Testing Frameworks

In [0]:
df.tail()

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes,id,age_normalized
99995,Male,19.0,0,0,No Info,27.32,6.5,140,0,12495,-1.016388
99996,Female,80.0,1,0,never,25.68,5.0,85,0,12496,1.692695
99997,Female,31.0,0,0,No Info,30.27,6.0,160,0,12497,-0.483454
99998,Female,80.0,0,0,No Info,28.20,8.8,159,1,12498,1.692695
99999,Male,37.0,0,0,never,27.90,6.2,85,0,12499,-0.216987


In [0]:
df_numeric.printSchema()

root
 |-- age: double (nullable = true)
 |-- hypertension: long (nullable = true)
 |-- heart_disease: long (nullable = true)
 |-- bmi: double (nullable = true)
 |-- HbA1c_level: double (nullable = true)
 |-- blood_glucose_level: long (nullable = true)
 |-- Diabetes_binary: long (nullable = true)
 |-- id: long (nullable = true)
 |-- gender_indexed: double (nullable = false)
 |-- smoking_history_indexed: double (nullable = false)



In [0]:
df_numeric.show(5)

+----+------------+-------------+-----+-----------+-------------------+---------------+-----+--------------+-----------------------+
| age|hypertension|heart_disease|  bmi|HbA1c_level|blood_glucose_level|Diabetes_binary|   id|gender_indexed|smoking_history_indexed|
+----+------------+-------------+-----+-----------+-------------------+---------------+-----+--------------+-----------------------+
|44.0|           0|            0|42.25|        6.2|                155|              0|25000|           0.0|                    3.0|
|49.0|           0|            0| 27.6|        6.0|                126|              0|25001|           0.0|                    2.0|
|58.0|           0|            0|38.24|        5.8|                 85|              0|25002|           0.0|                    1.0|
|57.0|           1|            0|22.95|        6.0|                 90|              0|25003|           0.0|                    1.0|
|66.0|           0|            0|33.76|        5.7|                10

In [0]:
df_numeric.columns

['age',
 'hypertension',
 'heart_disease',
 'bmi',
 'HbA1c_level',
 'blood_glucose_level',
 'Diabetes_binary',
 'id',
 'gender_indexed',
 'smoking_history_indexed']

In [0]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   gender               100000 non-null  object 
 1   age                  100000 non-null  float64
 2   hypertension         100000 non-null  int64  
 3   heart_disease        100000 non-null  int64  
 4   smoking_history      100000 non-null  object 
 5   bmi                  100000 non-null  float64
 6   HbA1c_level          100000 non-null  float64
 7   blood_glucose_level  100000 non-null  int64  
 8   diabetes             100000 non-null  int64  
 9   id                   100000 non-null  int64  
 10  age_normalized       100000 non-null  float64
dtypes: float64(4), int64(5), object(2)
memory usage: 8.4+ MB


In [0]:
import unittest
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType, LongType, StringType
from pyspark.sql.functions import col, sum, min

class TestDataValidation(unittest.TestCase):
    """
    Unit tests for schema validation, missing values, and non-negative values in PySpark DataFrames.
    """

    @classmethod
    def setUpClass(cls):
        """
        Set up shared resources for the tests.
        """
        # Load the test DataFrame (assume a table named 'diabetes' is present)
       #cls.df = spark.read.format("delta").table("diabetes").select(
       #     'id', 'Diabetes_binary', 'HighBP', 'BMI', 'Smoker', 'Stroke',
       #     'HeartDiseaseorAttack', 'Age'
       # )

        cls.df = spark.read.format("delta").table("diabetes_new").select(
            'id', 'diabetes', 'age', 'hypertension', 'heart_disease', 'bmi',
             'HbA1c_level', 'blood_glucose_level',  'gender', 'smoking_history'
            
        )

    def test_validate_schema(self):
        """
        Test if the DataFrame schema matches the expected schema.
        """
        expected_schema = StructType([
            StructField("id", LongType(), True),
            StructField("diabetes", LongType(), True),
            StructField("age", DoubleType(), True),
            StructField("hypertension", LongType(), True),
            StructField("heart_disease", LongType(), True),
            StructField("bmi", DoubleType(), True),
            StructField("HbA1c_level", DoubleType(), True),
            StructField("blood_glucose_level", LongType(), True),
            StructField("gender", StringType(), True),
            StructField("smoking_history", StringType(), True),
        ])
        actual_schema = self.df.schema
        self.assertEqual(
            actual_schema, expected_schema,
            f"Schema validation failed.\nExpected: {expected_schema}\nActual: {actual_schema}"
        )

    def test_validate_no_missing_values(self):
        """
        Test that there are no missing (null) values in the DataFrame.
        """
        missing_values = self.df.agg(*[
            sum(col(c).isNull().cast("int")).alias(c) for c in self.df.columns
        ]).collect()[0].asDict()

        missing_columns = {col: missing_values[col] for col in self.df.columns if missing_values[col] > 0}
        self.assertFalse(
            missing_columns,
            f"Missing values found in the following columns: {missing_columns}"
        )

    def test_validate_non_negative_values(self):
        """
        Test that all numeric columns in the DataFrame contain non-negative values (>= 0).
        """
        # Get only numeric columns
        numeric_columns = [f.name for f in self.df.schema.fields 
                          if isinstance(f.dataType, (DoubleType, IntegerType, LongType))]
        
        negative_values = self.df.agg(*[
            min(col(c)).alias(c) for c in numeric_columns
        ]).collect()[0].asDict()

        negative_columns = {col: negative_values[col] for col in numeric_columns if negative_values[col] < 0}
        self.assertFalse(
            negative_columns,
            f"Negative values found in the following columns: {negative_columns}"
        )


# Run the tests
suite = unittest.TestLoader().loadTestsFromTestCase(TestDataValidation)
unittest.TextTestRunner().run(suite)

...
----------------------------------------------------------------------
Ran 3 tests in 1.193s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>

Todos os 3 testes passaram:

✅ test_validate_schema - Schema validado corretamente
✅ test_validate_no_missing_values - Sem valores nulos
✅ test_validate_non_negative_values - Valores não-negativos nas colunas numéricas